### Develop extractor to retrieve daily water readings.

In [ ]:
import re
from datetime import datetime
from io import BytesIO

import pytesseract
import requests
from bs4 import BeautifulSoup
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [2]:
SCRIPPS_URL = 'https://shorestations.ucsd.edu/about/scripps-pier/'
headers = {"User-Agent": "Mozilla/5.0"}

In [12]:
SCRIPPS_URL = 'https://shorestations.ucsd.edu/about/scripps-pier/'
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(SCRIPPS_URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
for tag in soup(["script", "style", "nav", "footer", "header"]):
    tag.decompose()
clean_text = soup.get_text(separator="\n", strip=True)
clean_text

"Scripps Pier, La Jolla | Shore Stations\nToday's Sea SURFACE Temperature\nat Scripps Pier in La Jolla:\nNote: sometimes the daily temperature is not taken\n*Scripps Pier daily temperature is adjusted to account for the sampling time of day\nToday's Sea BOTTOM Temperature\nat Scripps Pier in La Jolla:\nNote: sometimes the daily temperature is not taken\n*Scripps Pier daily temperature is adjusted to account for the sampling time of day\nCLICK\nHERE\nTO ACCESS Scripps pier, la jolla DATA FROM 1916-CURRENT\nSee how the daily temperature reading compares to the long term average:\nsurface climatology\nbottom climatology\ndaily vs long-term - surface\nlast 30 days - surface\ndaily vs long-term - bottom\nlast 30 days - bottom\n▸\nClick here for information on how to read and interpret the figure above\nIn the plot above, you can find the most recent temperature data written out in the top left corner, with the sampling date and Sea Surface Temperature (SST) in degrees Celsius and degrees Fa

In [3]:
SURFACE_IMG = "https://shorestations.ucsd.edu/plots/SIO_surf_temp_now.png"
BOTTOM_IMG = "https://shorestations.ucsd.edu/plots/SIO_bot_temp_now.png"

In [7]:
def ocr_img_text(img_url: str)->str:
    """
    convert the text in the image into text
    """
    resp = requests.get(img_url, headers=headers, timeout=15)
    resp.raise_for_status()
    img = Image.open(BytesIO(resp.content))

    img = img.convert("L")
    img = img.resize((img.width * 4, img.height * 4), Image.LANCZOS)
 
    return pytesseract.image_to_string(img)

In [9]:
img_text = ocr_img_text(SURFACE_IMG)

In [10]:
img_text

'O\n24.3 C\n\nMeasured on:\n\nJuly 27, 2026\nat 12:37\n\n'

In [13]:
def parse_temp(img_text: str)->str:
    """
    parse out the temperature from the image text
    """
    c_match = re.search(r"(-?\d+\.?\d*)\s*°?\s*C", img_text)

    temp_c = float(c_match.group(1)) if c_match else None

    return temp_c

In [12]:
parse_temp(img_text)

24.3

In [14]:
def parse_date(img_text: str)->str:
    DATE_PATTERNS = [
    r"\d{1,2}/\d{1,2}/\d{2,4}",       # 7/27/2026 or 07/27/26
    r"\d{4}-\d{1,2}-\d{1,2}",         # 2026-07-27
    r"[A-Za-z]{3,9}\.?\s+\d{1,2},?\s+\d{4}",  # July 27, 2026 / Jul. 27 2026
]
    for pattern in DATE_PATTERNS:
        match = re.search(pattern, img_text)
        if match:
            return match.group(0)
    return None
    

In [16]:
date_text = parse_date(img_text)
date_text

'July 27, 2026'

In [18]:
surface_img_text = ocr_img_text(SURFACE_IMG)

surface_temp = parse_temp(surface_img_text)

date_text =  parse_date(surface_img_text)
date_today = datetime.strptime(date_text, '%B %d, %Y')

bottom_img_text = ocr_img_text(BOTTOM_IMG)
bottom_temp = parse_temp(bottom_img_text)

print('date = ', date_today)
print('surface temp = ', surface_temp)
print('bottom temp = ', bottom_temp)

date =  2026-07-27 00:00:00
surface temp =  24.3
bottom temp =  23.9
